# Foraging Cache — DuckDB Query Examples

The production parquet database lives on S3:
- `s3://aind-scratch-data/aind-dynamic-foraging-cache/session_table.parquet` — flat session table (one row per session)
- `.../trial_table/` — Hive-partitioned by `subject_id` (one row per trial)
- `.../event_table/` — Hive-partitioned by `subject_id` (one row per behavioral event)

DuckDB reads `s3://` natively and the cache bucket is **public** — no AWS credentials or setup are required. The paths (`SESSION_DB`, `TRIAL_DB`, `EVENT_DB`) are importable from `aind_dynamic_foraging_data_utils.foraging_cache`; reassign them to a local directory to query a local build instead.

**Primary pattern**: filter the session table, then JOIN to the trial/event tables so every row carries the session-level metadata (subject, date, task, `foraging_eff`, …) alongside the trial data.

Three options when reading the partitioned tables:
- `hive_partitioning=true` — partition-level pruning on `subject_id`
- `union_by_name=true` — merges column schemas across files (different NWB readers produce different column sets; missing columns fill with NULL)
- `CAST(subject_id AS VARCHAR)` — DuckDB infers the `subject_id=<n>` directory name as BIGINT; cast to VARCHAR to compare against the session table

In [1]:
import time

import duckdb

from aind_dynamic_foraging_data_utils.foraging_cache import SESSION_DB, TRIAL_DB, EVENT_DB

# SESSION_DB / TRIAL_DB / EVENT_DB point at the public production cache on S3.
# Reassign them to a local directory to query a local build instead.

# Reusable snippets - union_by_name handles schema differences across NWB reader paths.
READ_TRIALS = f"read_parquet('{TRIAL_DB}/**/*.parquet', hive_partitioning=true, union_by_name=true)"
READ_EVENTS = f"read_parquet('{EVENT_DB}/**/*.parquet', hive_partitioning=true, union_by_name=true)"

## 1. Database at a glance

A quick orientation to the whole cache. Session/mice counts come from one cheap scan of the flat session table; total trials and the timed load scan the partitioned trial table.

In [2]:
# Sessions, mice, and date span — one scan of the flat session table
duckdb.sql(f"""
    SELECT
        COUNT(*)                   AS total_sessions,
        COUNT(DISTINCT subject_id) AS total_mice,
        MIN(session_date)          AS first_session,
        MAX(session_date)          AS last_session
    FROM read_parquet('{SESSION_DB}')
""").df()

,total_sessions,total_mice,first_session,last_session
0,23865,902,2019-06-25,2026-06-03


In [3]:
# Distribution of data sources (which NWB route produced each session)
duckdb.sql(f"""
    SELECT nwb_data_source,
           COUNT(*)                   AS n_sessions,
           COUNT(DISTINCT subject_id) AS n_mice
    FROM read_parquet('{SESSION_DB}')
    GROUP BY nwb_data_source
    ORDER BY n_sessions DESC
""").df()

,nwb_data_source,n_sessions,n_mice
0,co_asset,15801,657
1,bpod_s3,4297,157
2,bonsai_s3,3767,343


In [4]:
# Total trials across the entire database (scans the partitioned trial table)
duckdb.sql(f"SELECT COUNT(*) AS total_trials FROM {READ_TRIALS}").df()

,total_trials
0,12521841


### Load the 5-column behavioral projection for **every** trial, and time it

Column projection (choice / reward / reward-probabilities) keeps this fast and light even over the full ~12.5M-trial database on S3 — the normal starting point for a population analysis.

In [5]:
t0 = time.time()
df_5col = duckdb.sql(f"""
    SELECT session_id, animal_response, earned_reward,
           reward_probabilityL, reward_probabilityR
    FROM {READ_TRIALS}
""").df()
elapsed = time.time() - t0

print(f"Loaded {len(df_5col):,} trials x {df_5col.shape[1]} columns "
      f"from {df_5col['session_id'].nunique():,} sessions "
      f"in {elapsed:.1f}s ({df_5col.memory_usage(deep=True).sum() / 1e9:.2f} GB in memory)")
df_5col.head()

Loaded 12,521,841 trials x 5 columns from 23,582 sessions in 8.4s (1.22 GB in memory)


,session_id,animal_response,earned_reward,reward_probabilityL,reward_probabilityR
0,447921_2019-09-11_135646,2.0,False,0.0,1.0
1,447921_2019-09-11_135646,2.0,False,0.0,1.0
2,447921_2019-09-11_135646,1.0,True,0.0,1.0
3,447921_2019-09-11_135646,1.0,True,0.0,1.0
4,447921_2019-09-11_135646,1.0,True,0.0,1.0


## 2. DuckDB basics (new to DuckDB?)

DuckDB is an in-process SQL engine that queries parquet (local or `s3://`) directly — no server and no separate load step. Essentials:
- `duckdb.sql(query)` returns a **lazy relation**; nothing runs until you materialise it with `.df()` (→ pandas), `.fetchall()`, or by displaying it.
- Read parquet with `read_parquet('…/**/*.parquet', hive_partitioning=true, union_by_name=true)` — use the `READ_TRIALS` / `READ_EVENTS` snippets defined above.
- Only the columns and rows you ask for are read from S3 (projection + filter *pushdown*), so **select the columns you need** instead of `SELECT *`.

In [ ]:
# List every column (name + type) WITHOUT reading any rows: DESCRIBE
duckdb.sql(f"DESCRIBE SELECT * FROM read_parquet('{SESSION_DB}')").df()

In [ ]:
# The trial table has ~100 columns (the union of all NWB readers). Count and list them.
trial_cols = duckdb.sql(f"DESCRIBE SELECT * FROM {READ_TRIALS}").df()
print(f"{len(trial_cols)} columns in the trial table")
trial_cols[["column_name", "column_type"]].head(25)

In [ ]:
# Column names as a plain Python list (schema only, no scan) — handy when building queries
duckdb.sql(f"SELECT * FROM {READ_EVENTS}").columns

In [ ]:
# Peek at a few rows; LIMIT avoids scanning the whole table
duckdb.sql(f"SELECT * FROM read_parquet('{SESSION_DB}') LIMIT 3").df()

In [ ]:
# SUMMARIZE = one-shot column stats (min / max / avg / std / approx-unique / null %, percentiles)
duckdb.sql(f"""
    SUMMARIZE SELECT animal_response, earned_reward, reward_probabilityL, reward_probabilityR
    FROM {READ_TRIALS}
""").df()

In [ ]:
# DuckDB can query an existing pandas DataFrame by its variable name (no copy) —
# so you can mix SQL and pandas freely.
recent = duckdb.sql(f"SELECT * FROM read_parquet('{SESSION_DB}') LIMIT 500").df()
duckdb.sql("SELECT task, COUNT(*) AS n FROM recent GROUP BY task ORDER BY n DESC LIMIT 5").df()

## 3. Browse the session table

In [6]:
duckdb.sql(f"""
    SELECT _session_id, subject_id, session_date, task, finished_trials, foraging_eff
    FROM read_parquet('{SESSION_DB}')
    ORDER BY session_date DESC
    LIMIT 10
""").df()

,_session_id,subject_id,session_date,task,finished_trials,foraging_eff
0,854145_2026-06-03_131806,854145,2026-06-03,Uncoupled Baiting,413.0,0.684090
1,844917_2026-06-03_125658,844917,2026-06-03,Uncoupled Baiting,304.0,0.574562
2,844634_2026-06-03_122723,844634,2026-06-03,Uncoupled Baiting,179.0,0.797416
3,845710_2026-06-03_122619,845710,2026-06-03,Uncoupled Baiting,482.0,0.695470
4,854147_2026-06-03_122510,854147,2026-06-03,Uncoupled Baiting,464.0,0.596816
5,853577_2026-06-03_103236,853577,2026-06-03,Uncoupled Without Baiting,347.0,0.731159
6,853578_2026-06-03_103217,853578,2026-06-03,Uncoupled Without Baiting,523.0,0.703180
7,844636_2026-06-03_103213,844636,2026-06-03,Uncoupled Baiting,469.0,0.645705
8,849570_2026-06-03_102934,849570,2026-06-03,Uncoupled Without Baiting,499.0,0.710744
9,844920_2026-06-03_102832,844920,2026-06-03,Uncoupled Baiting,523.0,0.780315


## 4. Primary use case — filter sessions -> trials with session keys merged

Filter the session table however you like, then JOIN to the trial table so every trial row already carries the session-level metadata.

In [7]:
df_trials = duckdb.sql(f"""
    WITH sel AS (
        SELECT _session_id, subject_id, session_date, task, foraging_eff
        FROM read_parquet('{SESSION_DB}')
        WHERE task LIKE '%Uncoupled%'
          AND foraging_eff > 0.8
    )
    SELECT
        s.subject_id,
        s.session_date,
        t.session_id,
        t.trial,
        t.animal_response,
        t.earned_reward,
        t.reward_probabilityL,
        t.reward_probabilityR,
        t.rewarded_historyL,
        t.rewarded_historyR
    FROM {READ_TRIALS} t
    JOIN sel s ON t.session_id = s._session_id
    WHERE CAST(t.subject_id AS VARCHAR) IN (SELECT subject_id FROM sel)
    ORDER BY s.subject_id, s.session_date
""").df()

print(f"{len(df_trials):,} trials from {df_trials['session_id'].nunique():,} sessions")
df_trials.head(10)

1,274,741 trials from 2,812 sessions


,subject_id,session_date,session_id,trial,animal_response,earned_reward,reward_probabilityL,reward_probabilityR,rewarded_historyL,rewarded_historyR
0,639872,2022-11-29,639872_2022-11-29_130622,NaN,0.0,False,0.3,0.05,False,False
1,639872,2022-11-29,639872_2022-11-29_130622,NaN,0.0,False,0.3,0.05,False,False
2,639872,2022-11-29,639872_2022-11-29_130622,NaN,2.0,False,0.3,0.05,False,False
3,639872,2022-11-29,639872_2022-11-29_130622,NaN,2.0,False,0.3,0.05,False,False
4,639872,2022-11-29,639872_2022-11-29_130622,NaN,0.0,False,0.3,0.05,False,False
5,639872,2022-11-29,639872_2022-11-29_130622,NaN,0.0,False,0.3,0.05,False,False
6,639872,2022-11-29,639872_2022-11-29_130622,NaN,2.0,False,0.3,0.05,False,False
7,639872,2022-11-29,639872_2022-11-29_130622,NaN,2.0,False,0.3,0.05,False,False
8,639872,2022-11-29,639872_2022-11-29_130622,NaN,0.0,True,0.3,0.05,True,False
9,639872,2022-11-29,639872_2022-11-29_130622,NaN,0.0,False,0.3,0.05,False,False


## 5. Same pattern for events

In [8]:
df_events = duckdb.sql(f"""
    WITH sel AS (
        SELECT _session_id, subject_id, session_date, task, foraging_eff
        FROM read_parquet('{SESSION_DB}')
        WHERE task LIKE '%Uncoupled%'
          AND foraging_eff > 0.8
    )
    SELECT
        s.subject_id,
        s.session_date,
        s.task,
        e.session_id,
        e.timestamps,
        e.event,
        e.data
    FROM {READ_EVENTS} e
    JOIN sel s ON e.session_id = s._session_id
    WHERE CAST(e.subject_id AS VARCHAR) IN (SELECT subject_id FROM sel)
    ORDER BY s.subject_id, s.session_date, e.timestamps
""").df()

print(f"{len(df_events):,} events from {df_events['session_id'].nunique():,} sessions")
print(f"Event types: {sorted(df_events['event'].unique())}")
df_events.head(10)

13,402,186 events from 2,812 sessions
Event types: ['goCue_start_time', 'left_lick_time', 'left_reward_delivery_time', 'optogenetics_time', 'right_lick_time', 'right_reward_delivery_time']


,subject_id,session_date,task,session_id,timestamps,event,data
0,639872,2022-11-29,Uncoupled Without Baiting,639872_2022-11-29_130622,6.2210,left_lick_time,1.0
1,639872,2022-11-29,Uncoupled Without Baiting,639872_2022-11-29_130622,13.0659,left_lick_time,1.0
2,639872,2022-11-29,Uncoupled Without Baiting,639872_2022-11-29_130622,13.2830,left_lick_time,1.0
3,639872,2022-11-29,Uncoupled Without Baiting,639872_2022-11-29_130622,33.7682,left_lick_time,1.0
4,639872,2022-11-29,Uncoupled Without Baiting,639872_2022-11-29_130622,33.9585,left_lick_time,1.0
5,639872,2022-11-29,Uncoupled Without Baiting,639872_2022-11-29_130622,38.3667,left_lick_time,1.0
6,639872,2022-11-29,Uncoupled Without Baiting,639872_2022-11-29_130622,38.5555,left_lick_time,1.0
7,639872,2022-11-29,Uncoupled Without Baiting,639872_2022-11-29_130622,57.7755,left_lick_time,1.0
8,639872,2022-11-29,Uncoupled Without Baiting,639872_2022-11-29_130622,57.7925,left_reward_delivery_time,1.0
9,639872,2022-11-29,Uncoupled Without Baiting,639872_2022-11-29_130622,58.1000,left_lick_time,1.0


## 6. Aggregate across sessions — all in SQL

In [9]:
duckdb.sql(f"""
    WITH sel AS (
        SELECT _session_id, subject_id, session_date, task, foraging_eff
        FROM read_parquet('{SESSION_DB}')
        WHERE task LIKE '%Uncoupled%'
          AND foraging_eff > 0.8
    )
    SELECT
        s.subject_id,
        COUNT(DISTINCT s._session_id)  AS n_sessions,
        COUNT(*)                       AS n_trials,
        AVG(t.earned_reward::DOUBLE)   AS mean_reward_rate,
        AVG(s.foraging_eff)            AS mean_foraging_eff
    FROM {READ_TRIALS} t
    JOIN sel s ON t.session_id = s._session_id
    WHERE CAST(t.subject_id AS VARCHAR) IN (SELECT subject_id FROM sel)
    GROUP BY s.subject_id
    ORDER BY mean_foraging_eff DESC
""").df()

,subject_id,n_sessions,n_trials,mean_reward_rate,mean_foraging_eff
0,766556,1,520,0.411538,1.052861
1,816995,4,44,0.659091,1.019733
2,707524,1,188,0.377660,1.002825
3,827183,3,32,0.781250,0.995615
4,812300,1,504,0.422619,0.985512
...,...,...,...,...,...
549,756566,2,1082,0.506470,0.802106
550,648585,2,1612,0.571340,0.800938
551,796263,1,604,0.503311,0.800842
552,781477,1,460,0.463043,0.800763
